# Import required libraries

In [1]:
import operator
import functools
import pandas as pd
from PIL import Image
from pydantic import BaseModel

# typing decorators
from typing import List, Tuple, Dict, Any, Sequence, Annotated, Literal
from typing_extensions import TypedDict

# langchain packages
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.output_parsers import (StrOutputParser, 
                                            JsonOutputParser)
from langchain_core.prompts import ChatPromptTemplate


# langgraph packages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, create_react_agent

# Transformer packages
from transformers import (AutoModelForImageClassification, 
                            AutoImageProcessor)


# Load custom defined packages
from models import (GenderPredictionFromPreTrainedModel, 
                    AgePredictionForPretrainedModel)

/Users/kavisanthoshkumar/Documents/GenerativeAIusingAWS/healthcare_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Getting Random Patient Details

In [2]:
# Load patient dataset
updated_patient_df = pd.read_csv('datasets/input/processed/updated_patient_df.csv')
updated_patient_df.drop('Unnamed: 0', axis = 1, inplace = True)
updated_patient_df.head(10)


# Pick a Random Patient: A female between 20 and 29 and with Pneumonia as Positive 
selected_patients = updated_patient_df[
    (updated_patient_df['Gender'] == 'Female') & \
        (updated_patient_df['Age'].between(20, 29)) & \
            (updated_patient_df['Difficulty Breathing'] == 'Yes') & \
                (updated_patient_df['Outcome Variable'] == 'Positive')]

# Choosing first patient details as our subject
selected_patients = (selected_patients
                                .reset_index(drop = True)
                                .iloc[0, :])

selected_patients


Disease                     Influenza
Fever                             Yes
Cough                             Yes
Fatigue                           Yes
Difficulty Breathing              Yes
Age                                25
Gender                         Female
Blood Pressure                 Normal
Cholesterol Level              Normal
Outcome Variable             Positive
First_Name                     Amrita
Last_Name                        Nair
Patient_ID              PAT1000000007
Full_Name                 Amrita_Nair
Name: 0, dtype: object

In [3]:
def patient_verification_tool(image_path, updated_patient_df, selected_patients):

    # Gender Prediction
    image = Image.open(image_path)

    # Load PreTrained Model - Gender
    gender_processor = AutoImageProcessor.from_pretrained("rizvandwiki/gender-classification")
    gender_model = AutoModelForImageClassification.from_pretrained("rizvandwiki/gender-classification")

    # Generate Gender Classification Predictions 
    predicted_gender = GenderPredictionFromPreTrainedModel(image, gender_processor, gender_model)

    # Load PreTrained Model - Age Group Prediction
    age_processor = AutoImageProcessor.from_pretrained("nateraw/vit-age-classifier")
    age_model = AutoModelForImageClassification.from_pretrained("nateraw/vit-age-classifier")

    # Generate Image Classification Predictions
    predicted_age_group = AgePredictionForPretrainedModel(image, age_processor, age_model)

    return predicted_gender, predicted_age_group


patient_verification_tool(
    image_path = 'datasets/FaceID/image1.png', 
    updated_patient_df = None, 
    selected_patients = None
)

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 20599.70it/s]


Gender Prediction using Pre-Trained Model : female


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 17373.11it/s]


Age Prediction to the Class: 20-29


('female', '20-29')